# Global rotations on colour codes

This tutorial reproduces the **ideal (noiseless) simulation** of Fig. 4a from
[Bluvstein et al., Nature (2026)](https://www.nature.com/articles/s41586-025-09848-5):
subjecting quantum error-correction codes of different dimensionality to a
**global $R_Z(\varphi)$ rotation**, then measuring the logical $X$ projection
and X-stabilizer revival.

We use **Tsim** to prepare $|+_L\rangle$ (hypercube encoder for $[[7,1,3]]$, Stim
graph-state synthesis for $[[15,1,3]]$). The rotation sweep uses exact state
vectors, which is fast for $n \le 15$ and matches the paper's noiseless logical
response. It is **not** the experimental lookup-table decoder or postselection
pipeline.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tsim import Circuit

from utils.global_rotation import (
    CODE_7_PLUS_L,
    GATE_LABELS_DEG,
    build_systems,
    compute_curve,
    reed_muller_prep_circuit,
    verify_encoding_flows,
    verify_steane_plus_l_flows,
)

## Physical setup

Fig. 4a studies

$$|+_L\rangle \;\xrightarrow{\; R_Z(\varphi)^{\otimes n}\;}\; \text{measure } X_L \text{ and stabilizers}$$

for four configurations:

* **2D colour code** — $[[7,1,3]]$ Steane code
* **3D colour code** — $[[15,1,3]]$ quantum Reed–Muller code
* **Unentangled** — $|+\rangle^{\otimes 15}$ analysed with the same 3D observables
* **3D negative stabilizer** — Reed–Muller state with flipped Z-check signs

Encoded 2D/3D codes show **X-stabilizer revivals** at Clifford angles ($0^\circ,
\pm 90^\circ, \ldots$). The 3D Reed–Muller code has an additional revival at
$\pm 45^\circ$ (transversal $T$), where $\langle X_L\rangle = 1/\sqrt{2}$.
Unentangled qubits analysed with the same 15-body observables revive only near
$\pm 180^\circ$.

This notebook is an **ideal noiseless** tutorial calculation. The experimental
Fig. 4a curves additionally use lookup-table decoding, postselection, and
purity normalization (Methods).

## Hypercube encoding of $|+_L\rangle$ for $[[7,1,3]]$

The QuEra codes belong to the Reed–Muller family (Extended Data Fig. 10a). For
the 2D Steane code the **fourth CZ layer is turned off**. Qubit 6 starts in
$|+\rangle$; qubits 0–5 in $|0\rangle$.

In [ ]:
encoding = Circuit(CODE_7_PLUS_L)
encoding.diagram("timeline-svg", height=320)

Verify stabilizers and logical operators with Stim's `flow_generators()`, as
suggested in [issue #119](https://github.com/QuEraComputing/tsim/issues/119):

In [ ]:
encoding_flows = verify_encoding_flows(CODE_7_PLUS_L)
prep_flows = verify_steane_plus_l_flows()
rm_flows = reed_muller_prep_circuit().flow_generators()

print(f"{len(encoding_flows)} flows on the Clifford encoding circuit")
print(f"{len(prep_flows)} flows on the full $[[7,1,3]]$ $|+_L\\rangle$ preparation")
print(f"{len(rm_flows)} flows on the $[[15,1,3]]$ preparation")
for flow in encoding_flows[:4]:
    print(flow)

## Global rotation sweep

For each $\varphi/\pi \in [-1, 1]$, we apply global $R_Z(\varphi)$ and evaluate

1. $\langle X_L\rangle$ (logical $X$ projection), and
2. mean $|\langle S_j^X\rangle|$ over X-type stabilizers (normalized to unit peak).

Global $R_Z$ disturbs X-checks while leaving Z-checks invariant; the bottom panel
tracks X-stabilizer revivals, matching the Fig. 4a convention.

In [ ]:
N_ANGLES = 37
angles_deg = np.linspace(-180, 180, N_ANGLES)

systems = build_systems()
curves = {system.label: compute_curve(system, angles_deg) for system in systems}

## Fig. 4a (ideal simulation)

In [ ]:
plot_order = [
    "2D colour ($[[7,1,3]]$)",
    "3D colour ($[[15,1,3]]$)",
    "Unentangled",
    "3D negative stabilizer",
]
colors = ["C0", "C3", "C2", "C1"]

fig, axes = plt.subplots(2, 1, figsize=(8, 5.5), sharex=True, constrained_layout=True)

for label, color in zip(plot_order, colors):
    curve = curves[label]
    axes[0].plot(curve.angles_deg, curve.logical_x, "-", color=color, label=label)
    if label != "Unentangled":
        axes[1].plot(curve.angles_deg, curve.stabilizer_abs, "-", color=color, label=label)

axes[0].set_ylabel(r"Logical $X$ projection")
axes[0].set_ylim(-1.05, 1.05)
axes[0].axhline(0, color="0.85", lw=0.8)
axes[0].legend(loc="lower center", ncol=2, fontsize=8, frameon=False)

axes[1].set_ylabel("Normalized abs. stabilizer")
axes[1].set_xlabel(r"Global $\varphi\;(^{\circ})$")
axes[1].set_ylim(-0.05, 1.05)
axes[1].legend(loc="lower center", ncol=2, fontsize=8, frameon=False)

for ax in axes:
    ax.set_xlim(-185, 185)
    ax.set_xticks(list(GATE_LABELS_DEG.keys()))
    ax.set_xticklabels(list(GATE_LABELS_DEG.values()))

fig.suptitle("Fig. 4a — global $R_Z$ rotation (noiseless simulation)", y=1.02)
plt.show()

## What to notice

* **2D colour** — X-stabilizers revive at $0^\circ$ and $\pm 90^\circ$; logical
  $X$ is $+1$ at $0^\circ$ and $\approx 0$ at $\pm 90^\circ$.
* **3D colour** — same Clifford stabilizer revivals **plus** a logical-$X$ value
  $1/\sqrt{2}$ at $\pm 45^\circ$ (transversal $T$).
* **Unentangled** — $\langle X_L\rangle = \cos^{15}\varphi$; revivals only near
  $\pm 180^\circ$, not at Clifford angles.
* **Negative stabilizer** — wrong Z-check signs remove the 45° T feature while
  Clifford stabilizer revivals remain.

The $[[15,1,3]]$ state is prepared from verified stabilizer generators (Stim
graph-state synthesis). The paper's hypercube encoder (Extended Data Fig. 10a) is
shown and verified for $[[7,1,3]]$ above; the full 15-qubit hypercube circuit is
not in the supplementary Stim files.

## Summary

In this notebook we:

* built and verified the QuEra hypercube encoder for $|+_L\rangle$ on $[[7,1,3]]$,
* prepared $[[15,1,3]]$ $|+_L\rangle$ from its stabilizer generators,
* swept a global $R_Z$ rotation and computed logical-$X$ and X-stabilizer observables, and
* reproduced the qualitative structure of Fig. 4a for all four curves.